# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata

# Print out the dataset's name and description
print(f"{md.name}: {md.description}")
print(f"Date Published: {md.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We will print out key record sets and their fields, referencing everyone by their `@id`.

In [ ]:
# List all available record sets with their '@id' and fields
record_sets = md.recordSet
if not record_sets:
    print("No record sets defined in the toplevel metadata (import failed or missing). Trying md.sources...")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        fields = rs.get('field', [])
        print("  Fields:")
        for f in fields:
            print(f"    Field @id: {f['@id']} name: {f.get('name', '<unnamed>')}")
        print()
# If the record sets attribute is empty (some schemas store only under distribution -> fileObject)
if not record_sets:
    # Try to extract fileObjects via distribution->fileObject
    for dist in getattr(md, 'distribution', []):
        file_objs = getattr(dist, 'fileObject', [])
        for fo in file_objs:
            print(f"FileObject @id: {fo['@id']}, name: {fo.get('name', '<unnamed>')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, the record sets need to be discovered from schema.
# We'll get all available recordSet @id's for use. If missing, try typical names, e.g., /records or check fileObjects.
discovered_record_set_ids = []

if record_sets:
    discovered_record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # Try fileObjects as record sets (common fallback)
    for dist in getattr(md, 'distribution', []):
        for fo in getattr(dist, 'fileObject', []):
            discovered_record_set_ids.append(fo['@id'])
# If still empty, try to extract via dataset.records (schema resolves for us)
if not discovered_record_set_ids and hasattr(dataset, 'list_record_sets'):
    discovered_record_set_ids = dataset.list_record_sets()

print(f"Record Sets for this dataset:")
for rsid in discovered_record_set_ids:
    print(f"  - {rsid}")

# Now, extract all records from each available record set and load them as DataFrames
dataframes = {}
for record_set_id in discovered_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for recordSet {record_set_id}")
        else:
            print(f"No records for recordSet {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick the first valid DataFrame for demonstration
main_record_set_id = None
for rsid in discovered_record_set_ids:
    if rsid in dataframes:
        main_record_set_id = rsid
        break

if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Sample columns from {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No valid dataframes loaded. Please inspect the dataset.records structure.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Pick a numeric field for analysis - often mass spectrometry data has a MW (molecular weight) or Abundance column
# Try common column names
numerical_candidates = ['MW', 'mw', 'Molecular_Weight', 'Abundance', 'Normalized_Abundance', 'value']

numeric_field = None
if main_record_set_id:
    available = dataframes[main_record_set_id].columns
    for nc in numerical_candidates:
        if nc in available:
            numeric_field = nc
            print(f"Using numeric field: {nc}")
            break

    if numeric_field:
        # Ensure column is float
        df = dataframes[main_record_set_id].copy()
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as filter example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (75th percentile): {filtered_df.shape[0]} rows")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by protein description or accession if found
        group_field_candidates = ['Description', 'description', 'Accession', 'accession', 'Protein_Name']
        group_field = None
        for gf in group_field_candidates:
            if gf in df.columns:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set DataFrame found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the main numeric field and a boxplot by the grouping if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if we found a numeric field and have a DataFrame
if main_record_set_id and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If grouped, try a boxplot
    if 'group_field' in locals() and group_field:
        # Display at most top N categories for clarity
        top_cats = filtered_df[group_field].value_counts().head(10).index
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(top_cats)])
        plt.title(f'{numeric_field} by {group_field} (Top 10)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion
In this notebook, we loaded FAIR² Croissant metadata and records for the protein mass spectrometry dataset using `mlcroissant`. After reviewing its structure and extracting a main record set, we performed basic exploratory data analysis, filtering, normalization, grouping, and visualized the distribution of key numeric attributes. You can adapt the notebook to your use case, e.g., by choosing different fields or constructing more complex preprocessing or visualization pipelines.